# Rudy classical asteroseismic baseline (Stage 3)

This executed audit notebook summarizes T3.1-T3.9 from machine-readable products generated by `TA-S03-T01-RUDY-R001`. The authoritative computation is `workflows/run_stage3_classical.py`; this notebook does not hide an alternate analysis path. Raw FITS payloads remain in the external checksum-verified cache. No topological statistic or final-holdout object is used.

In [1]:
import json
from pathlib import Path

root = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
manifest = json.loads((root / 'results/stage3/run_manifest.json').read_text())
print(manifest['run_id'])
print(manifest['software_version'])
print(f"{len(manifest['inputs'])} checksum-sealed inputs; topology_used={manifest['topology_used']}")

TA-S03-T01-RUDY-R001
TopoAstero-0.4.0
21 checksum-sealed inputs; topology_used=False


## T3.1-T3.2: reproduction and independent pipeline

The Lightkurve path exactly follows the documented Q2/Q5/Q6/Q7 tutorial. The independent path uses minimum-BIC Harvey whitening and an FFT autocorrelation. Both paths share products, absolute timestamps, one-sided PSD convention, and frequency units.

In [2]:
import csv

with (root / 'results/stage3/rudy_baseline.csv').open(newline='') as handle:
    rows = list(csv.DictReader(handle))
by_pipeline = {}
for row in rows:
    by_pipeline.setdefault(row['pipeline'], {})[row['quantity']] = float(row['value'])
print(f"{'pipeline':24s} {'numax':>10s} {'dnu':>15s} {'mass':>15s} {'radius':>15s} {'logg':>15s}")
for pipeline, values in by_pipeline.items():
    print(f"{pipeline:24s} {values['numax']:10.4f} {values['dnu']:15.4f} {values['mass']:15.4f} {values['radius']:15.4f} {values['logg']:15.4f}")

pipeline                 numax             dnu            mass          radius           logg
lightkurve_acf2d         2145.0000      103.1121          1.0816          1.2291          4.2930
independent_harvey_acf   2231.5686      103.3662          1.2043          1.2718          4.3099


## T3.3-T3.8: observation model, echelle, modes, and uncertainty

The two-component Harvey-like model is selected by BIC. The absolute-time sampling window remains primary; compressed-quarter time is diagnostic only. All 42 published modes are audited under one fixed peak rule. The 500-resample segment bootstrap, branch half-range, and reference interval remain separate columns.

![Rudy spectrum](../reports/stage3/figures/rudy_power_spectrum.svg)

![Conventional echelle trials](../reports/stage3/figures/echelle_trials.svg)

![Mode audit](../reports/stage3/figures/mode_overlay_audit.svg)

In [3]:
summary = manifest['summary']
print(f"background={summary['background_model']}; modes={summary['mode_count']}; order offsets={summary['mode_order_offsets']}")
with (root / 'results/stage3/uncertainty_budget.csv').open(newline='') as handle:
    uncertainty = {row['quantity']: row for row in csv.DictReader(handle)}
for quantity in ('numax', 'dnu'):
    row = uncertainty[quantity]
    print(f"{quantity} uncertainty components (uHz): random={float(row['random_bootstrap_sigma']):.3f}, pipeline={float(row['pipeline_choice_half_range']):.3f}, reference={float(row['reference_interval_half_width']):.3f}")

background=BG-H2; modes=42; order offsets={'1': 29, '2': 13}
numax uncertainty components (uHz): random=93.777, pipeline=43.284, reference=6.500
dnu uncertainty components (uHz): random=0.096, pipeline=0.412, reference=0.027


## T3.9: TESS feasibility

Sectors are analyzed individually. One marginal candidate is not confirmation, ten sectors remain non-detections, and no combined detection is forced.

![TESS feasibility](../reports/stage3/figures/tess_sector_feasibility.svg)

In [4]:
print(summary['tess_classifications'])
print('combination decision: do_not_combine_for_detection_at_G3')

{'marginal_candidate_not_confirmation': 1, 'not_detected_individually': 10}
combination decision: do_not_combine_for_detection_at_G3


## G3 conclusion

The tutorial result is reproduced, independent discrepancies are traced and included as pipeline-choice uncertainty, PSD/background analytic tests pass, and TESS failures are retained. Gate G3 is therefore **PASS**, with the limitations recorded in `reports/stage3/G3_ACCEPTANCE.json`.